# Inspeção e pré-processamento do UNSW-NB15

Este notebook valida os arquivos oficiais de treino e teste. A imputação e a codificação serão ajustadas somente sobre o treino dentro do pipeline do notebook `06_lightgbm.ipynb`, evitando vazamento de dados.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
DIRETORIO_DADOS = Path("dados_unsw_nb15")
CAMINHO_TREINO = DIRETORIO_DADOS / "UNSW_NB15_training-set.parquet"
CAMINHO_TESTE = DIRETORIO_DADOS / "UNSW_NB15_testing-set.parquet"

arquivos_ausentes = [
    str(caminho) for caminho in (CAMINHO_TREINO, CAMINHO_TESTE)
    if not caminho.exists()
]
if arquivos_ausentes:
    raise FileNotFoundError(
        "Arquivos ausentes: " + ", ".join(arquivos_ausentes)
        + ". Execute primeiro: python 04_download_unsw_nb15.py"
    )

treino = pd.read_parquet(str(CAMINHO_TREINO))
teste = pd.read_parquet(str(CAMINHO_TESTE))

print("Treino:", treino.shape)
print("Teste:", teste.shape)
display(treino.head())

Treino: (175341, 36)
Teste: (82332, 36)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.121478,tcp,-,FIN,6,4,258,172,74.087486,14158.942383,...,0,0,1,1,0,0,0,0,Normal,0
1,0.649902,tcp,-,FIN,14,38,734,42014,78.473373,8395.112305,...,0,0,1,1,0,0,0,0,Normal,0
2,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,1572.271851,...,0,0,1,1,0,0,0,0,Normal,0
3,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,2740.178955,...,0,0,1,1,1,1,0,0,Normal,0
4,0.449454,tcp,-,FIN,10,6,534,268,33.373825,8561.499023,...,0,0,2,1,0,0,0,0,Normal,0


## Validação do esquema e do alvo

In [3]:
COLUNAS_OBRIGATORIAS = {"proto", "service", "state", "attack_cat", "label"}

for nome, dados in {"treino": treino, "teste": teste}.items():
    ausentes = COLUNAS_OBRIGATORIAS.difference(dados.columns)
    if ausentes:
        raise ValueError(f"Colunas ausentes em {nome}: {sorted(ausentes)}")
    rotulos_invalidos = set(dados["label"].dropna().unique()).difference({0, 1})
    if rotulos_invalidos:
        raise ValueError(f"Rótulos inválidos em {nome}: {rotulos_invalidos}")

if list(treino.columns) != list(teste.columns):
    raise ValueError("Treino e teste têm esquemas diferentes.")

print(f"Esquema validado: {len(treino.columns)} colunas.")
display(treino.dtypes.rename("dtype").to_frame())

Esquema validado: 36 colunas.


,dtype
dur,float32
proto,category
service,category
state,category
spkts,int16
dpkts,int16
sbytes,int32
dbytes,int32
rate,float32
sload,float32


In [4]:
def distribuicao_classes(dados):
    totais = dados["label"].value_counts().reindex([0, 1], fill_value=0)
    resultado = pd.DataFrame({
        "classe": ["Normal", "Ataque"],
        "quantidade": totais.to_numpy(),
    })
    resultado["percentual"] = (resultado["quantidade"] / len(dados) * 100).round(2)
    return resultado.set_index("classe")

print("Distribuição no treino:")
display(distribuicao_classes(treino))
print("Distribuição no teste:")
display(distribuicao_classes(teste))
print("Categorias de ataque no treino:")
display(treino["attack_cat"].value_counts(dropna=False).to_frame("quantidade"))

Distribuição no treino:


,quantidade,percentual
classe,,
Normal,56000,31.94
Ataque,119341,68.06


Distribuição no teste:


,quantidade,percentual
classe,,
Normal,37000,44.94
Ataque,45332,55.06


Categorias de ataque no treino:


,quantidade
attack_cat,
Normal,56000
Generic,40000
Exploits,33393
Fuzzers,18184
DoS,12264
Reconnaissance,10491
Analysis,2000
Backdoor,1746
Shellcode,1133


## Qualidade dos dados

In [5]:
def resumo_qualidade(dados):
    numericas = dados.select_dtypes(include="number")
    infinitos = pd.Series(0, index=dados.columns, dtype="int64")
    infinitos.loc[numericas.columns] = np.isinf(numericas).sum().astype("int64")
    return pd.DataFrame({
        "tipo": dados.dtypes.astype(str),
        "ausentes": dados.isna().sum(),
        "infinitos": infinitos,
        "valores_unicos": dados.nunique(dropna=False),
    })

display(resumo_qualidade(treino))
print("Linhas duplicadas no treino:", treino.duplicated().sum())
print("Linhas duplicadas no teste:", teste.duplicated().sum())

treino_unico = treino.drop_duplicates()
teste_unico = teste.drop_duplicates()
hashes_treino = set(pd.util.hash_pandas_object(treino_unico, index=False))
hashes_teste = pd.util.hash_pandas_object(teste_unico, index=False)
sobrepostas = hashes_teste.isin(hashes_treino).sum()
print("Linhas únicas do teste idênticas a linhas do treino:", sobrepostas)

,tipo,ausentes,infinitos,valores_unicos
dur,float32,0,0,74039
proto,category,0,0,133
service,category,0,0,13
state,category,0,0,9
spkts,int16,0,0,480
dpkts,int16,0,0,443
sbytes,int32,0,0,7214
dbytes,int32,0,0,6660
rate,float32,0,0,76794
sload,float32,0,0,80885


Linhas duplicadas no treino: 78519
Linhas duplicadas no teste: 32361
Linhas únicas do teste idênticas a linhas do treino: 1571


In [6]:
COLUNAS_CATEGORICAS = ["proto", "service", "state"]

for coluna in COLUNAS_CATEGORICAS:
    categorias_treino = set(treino[coluna].dropna().astype(str))
    categorias_teste = set(teste[coluna].dropna().astype(str))
    print(
        f"{coluna}: {len(categorias_treino)} categorias no treino; "
        f"{len(categorias_teste - categorias_treino)} aparecem somente no teste."
    )

proto: 133 categorias no treino; 0 aparecem somente no teste.
service: 13 categorias no treino; 0 aparecem somente no teste.
state: 9 categorias no treino; 2 aparecem somente no teste.


## Próximo passo

O notebook `06_lightgbm.ipynb` remove duplicatas exatas, exclui do teste cópias já presentes no treino, remove `attack_cat` das entradas e usa `label` como alvo. Valores ausentes, infinitos e categorias são tratados dentro de um pipeline ajustado exclusivamente no treino. Não é gerado um CSV one-hot global, pois isso aumentaria o uso de disco e poderia introduzir vazamento de informação.